In [1]:
# === 1. 导入库 ===
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterstats import zonal_stats
import pylandstats as pls
from tqdm import tqdm

In [2]:
# === 2. 定义参数和文件路径 ===

# -- 基本参数 --
YEARS = [2000, 2005, 2010, 2015, 2020, 2024]
HEX_GRID_PATH = 'buffer/mangrove_2020_500m_hex_grid.shp'
OUTPUT_CSV = 'final_environment.csv'

# -- 数据目录 --
LULC_DIR = 'LULC_convert'
NTL_DIR = 'NTL_clipped'
VI_DIR = 'VI' # 新增：植被指数目录
POP_DIR = 'POP' # 新增：人口数据目录

# -- LULC 类别定义 --
LULC_CLASSES = {
    1: 'cropland', 2: 'forest', 3: 'shrub', 4: 'grassland',
    5: 'water', 7: 'bare_land', 8: 'impervious_surface'
}

# -- 植被指数 (VI) 定义 (波段号) --
VI_BANDS = {
    'NDVI': 2, 'EVI': 3, 'MVI': 4, 'EMVI': 5, 'CMRI': 6, 'kNDVI': 7
}
MANGROVE_MASK_BAND = 1 # 红树林二值影像在波段1

# -- 景观指数定义 --
# 使用 pylandstats 支持的正确指标名称
LANDSCAPE_METRICS = [
    'proportion_of_landscape',  # 替代 percentage_of_landscape
    'number_of_patches', 
    'patch_density',
    'largest_patch_index', 
    'area_mn',  # 替代 mean_patch_area
    'edge_density'
]

# -- 坐标系定义 --
# 使用一个等面积投影来准确计算面积和密度
PROJECTED_CRS = 'ESRI:54009' # WGS 84 / NSIDC EASE-Grid 2.0 Global

print("所有参数已定义完毕。")

所有参数已定义完毕。


In [3]:
def calculate_landscape_metrics(geom, raster_path):
    """为单个几何图形内的栅格计算景观指数。"""
    try:
        with rasterio.open(raster_path) as src:
            # === 修改 1: 使用 255 作为边界外的 nodata，保留 0 作为背景 ===
            # filled=True 确保被掩膜区域被填充为 nodata 值
            LANDSCAPE_NODATA = 255
            out_image, out_transform = mask(src, [geom], crop=True, nodata=LANDSCAPE_NODATA, filled=True)
            
            # 获取第一个波段
            raw_band = out_image[0, :, :]
            
            # === 修改 2: 构建景观数组 ===
            # 0: 背景 (网格内的非红树林)
            # 1: 类别 (红树林)
            # 255: 无效 (网格外部)
            
            # 初始化为 0 (背景)
            mangrove_band = np.zeros_like(raw_band, dtype='uint8')
            
            # 标记红树林 (假设原始数据中 > 0 为红树林，且不是 nodata)
            # 注意排除掉我们刚刚填充的 LANDSCAPE_NODATA
            is_data = (raw_band != LANDSCAPE_NODATA)
            mangrove_band[is_data & (raw_band > 0)] = 1
            
            # 标记外部区域为 nodata
            mangrove_band[~is_data] = LANDSCAPE_NODATA
            
            # 检查是否有有效数据 (类别 1)
            if not np.any(mangrove_band == 1):
                return {metric: 0 for metric in LANDSCAPE_METRICS}
            
            # 获取像素分辨率
            pixel_res_x = abs(out_transform[0])
            pixel_res_y = abs(out_transform[4])
            
            # 根据坐标系判断分辨率
            if pixel_res_x < 1:
                # 地理坐标系(度),需要转换为米
                import math
                minx = out_transform[2]
                maxy = out_transform[5]
                center_lat = maxy - (mangrove_band.shape[0] * pixel_res_y) / 2
                meters_per_degree = 111320 * math.cos(math.radians(center_lat))
                pixel_resolution = pixel_res_x * meters_per_degree
            else:
                pixel_resolution = pixel_res_x
            
            # === 修改 3: 初始化 Landscape 时指定正确的 nodata ===
            ls = pls.Landscape(
                mangrove_band, 
                res=(pixel_resolution, pixel_resolution), 
                nodata=LANDSCAPE_NODATA  # 告诉 pylandstats 忽略 255，但包含 0
            )
            
            # 检查是否有类别1
            if 1 not in ls.classes:
                return {metric: 0 for metric in LANDSCAPE_METRICS}
            
            # 计算类别指标
            metrics_df = ls.compute_class_metrics_df(metrics=LANDSCAPE_METRICS)
            
            if metrics_df.empty:
                return {metric: 0 for metric in LANDSCAPE_METRICS}
            
            # 提取类别1的指标
            if 1 in metrics_df.index:
                result = metrics_df.loc[1].to_dict()
            elif 'class_val' in metrics_df.columns:
                class_1_row = metrics_df[metrics_df['class_val'] == 1]
                if class_1_row.empty:
                    return {metric: 0 for metric in LANDSCAPE_METRICS}
                result = class_1_row.iloc[0].to_dict()
            else:
                result = metrics_df.iloc[0].to_dict()
            
            # 只返回需要的指标
            return {k: result.get(k, 0) for k in LANDSCAPE_METRICS}
        
    except Exception as e:
        print(f"景观指数计算错误: {e}")
        import traceback
        traceback.print_exc()
        return {metric: np.nan for metric in LANDSCAPE_METRICS}

def calculate_conditional_vi_mean(geom, vi_raster_path, vi_band, mask_band_num):
    """在红树林掩膜下，为单个几何图形计算植被指数的均值。"""
    try:
        with rasterio.open(vi_raster_path) as src:
            out_image, out_transform = mask(src, [geom], crop=True, nodata=src.nodata, filled=False)
            
            mangrove_mask = out_image[mask_band_num - 1, :, :]
            vi_data = out_image[vi_band - 1, :, :]
            
            # 转换为普通数组
            if hasattr(mangrove_mask, 'filled'):
                mangrove_mask = mangrove_mask.filled(0)
            if hasattr(vi_data, 'filled'):
                vi_data = vi_data.filled(np.nan)
            
            # 只计算红树林区域的植被指数
            vi_data_masked = np.where(mangrove_mask == 1, vi_data, np.nan)
            
            if np.all(np.isnan(vi_data_masked)):
                return np.nan
            return np.nanmean(vi_data_masked)
    except Exception as e:
        print(f"植被指数计算错误: {e}")
        return np.nan

print("辅助计算函数已定义。")

辅助计算函数已定义。


In [4]:
# === 4. 主处理循环 ===

grid = gpd.read_file(HEX_GRID_PATH)
all_records = []
print(f"加载了 {len(grid)} 个六边形网格,开始处理...")

for year in tqdm(YEARS, desc="Processing Years"):
    lulc_path = os.path.join(LULC_DIR, f'CLCD_v01_{year}_wgs84_clipped.tif')
    ntl_path = os.path.join(NTL_DIR, f'NTL_{year}_clipped.tif')
    vi_path = os.path.join(VI_DIR, f'mangrove_{year}_90conf_indices.tif')
    pop_path = os.path.join(POP_DIR, f'chn_ppp_{year}_1km_Aggregated.tif')

    if not all(os.path.exists(p) for p in [lulc_path, ntl_path, vi_path, pop_path]):
        print(f"\n警告: {year} 年数据文件不完整,跳过。")
        continue

    # 读取栅格的CRS
    with rasterio.open(lulc_path) as src:
        raster_crs = src.crs
    
    with rasterio.open(vi_path) as src:
        vi_crs = src.crs
    
    # 将网格转换为不同栅格的坐标系
    grid_matched = grid.to_crs(raster_crs)  # 用于LULC、NTL和POP
    grid_vi = grid.to_crs(vi_crs)  # 用于VI和景观指数
    
    # 使用匹配坐标系的网格进行统计
    lulc_stats = zonal_stats(grid_matched, lulc_path, categorical=True)
    ntl_stats = zonal_stats(grid_matched, ntl_path, stats="mean")
    pop_stats = zonal_stats(grid_matched, pop_path, stats="sum")
    
    for i, (geom_lulc, geom_vi) in enumerate(tqdm(zip(grid_matched.geometry, grid_vi.geometry), 
                                                 desc=f"Hexagons {year}", 
                                                 leave=False, 
                                                 total=len(grid))):
        record = {'GridID': grid['GridID'][i], 'year': year}
        
        # 处理 LULC 统计
        lulc_dict = lulc_stats[i]
        if not lulc_dict:
            for cls, name in LULC_CLASSES.items():
                record[f'{name}_percent'] = 0
        else:
            total_pixels = sum(v for k, v in lulc_dict.items() if k is not None and isinstance(k, (int, float)))
            
            if total_pixels > 0:
                for cls, name in LULC_CLASSES.items():
                    pixel_count = lulc_dict.get(cls, 0)
                    record[f'{name}_percent'] = (pixel_count / total_pixels) * 100
            else:
                for cls, name in LULC_CLASSES.items():
                    record[f'{name}_percent'] = 0
        
        # 处理 NTL 和 POP
        record['ntl_mean'] = ntl_stats[i].get('mean') if ntl_stats[i] else np.nan
        record['pop_sum'] = pop_stats[i].get('sum') if pop_stats[i] else np.nan
        
        # 使用 VI 栅格坐标系计算景观指数
        lm_results = calculate_landscape_metrics(geom_vi, vi_path)
        record.update(lm_results)
        
        # 使用 VI 栅格坐标系计算植被指数
        for vi_name, vi_band_num in VI_BANDS.items():
            vi_mean = calculate_conditional_vi_mean(geom_vi, vi_path, vi_band_num, MANGROVE_MASK_BAND)
            record[f'{vi_name}_mean_mangrove'] = vi_mean
            
        all_records.append(record)

print("\n所有年份数据处理完成!")

加载了 2512 个六边形网格,开始处理...


Processing Years:   0%|          | 0/6 [00:00<?, ?it/s]c:\Users\jr\miniconda3\envs\geo\Lib\site-packages\rasterstats\io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(
Processing Years: 100%|██████████| 6/6 [10:50<00:00, 108.37s/it]


所有年份数据处理完成!


In [5]:
# === 5. 合并与整理数据 ===

if not all_records:
    print("警告: 未生成任何记录，无法继续。请检查输入文件和路径。")
else:
    results_df = pd.DataFrame(all_records)
    grid['lon'] = grid.geometry.centroid.x
    grid['lat'] = grid.geometry.centroid.y
    final_df = pd.merge(results_df, grid[['GridID', 'lon', 'lat']], on='GridID', how='left')

    id_cols = ['GridID', 'year', 'lon', 'lat']
    lulc_cols = sorted([c for c in final_df.columns if '_percent' in c])
    ntl_cols = ['ntl_mean']
    pop_cols = ['pop_sum']
    landscape_cols = sorted([c for c in final_df.columns if c in LANDSCAPE_METRICS])
    vi_cols = sorted([c for c in final_df.columns if '_mean_mangrove' in c])
    
    final_df = final_df[id_cols + lulc_cols + ntl_cols + pop_cols + landscape_cols + vi_cols]

    print("数据合并与整理完成。")
    print("\n最终DataFrame信息:")
    final_df.info()
    print("\n最终DataFrame预览:")
    print(final_df.head())

数据合并与整理完成。

最终DataFrame信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15072 entries, 0 to 15071
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   GridID                      15072 non-null  int64  
 1   year                        15072 non-null  int64  
 2   lon                         15072 non-null  float64
 3   lat                         15072 non-null  float64
 4   bare_land_percent           15072 non-null  float64
 5   cropland_percent            15072 non-null  float64
 6   forest_percent              15072 non-null  float64
 7   grassland_percent           15072 non-null  float64
 8   impervious_surface_percent  15072 non-null  float64
 9   shrub_percent               15072 non-null  float64
 10  water_percent               15072 non-null  float64
 11  ntl_mean                    14297 non-null  float64
 12  pop_sum                     8106 non-null   float64
 13  area

C:\Users\jr\AppData\Local\Temp\ipykernel_10504\3986136400.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid['lon'] = grid.geometry.centroid.x
C:\Users\jr\AppData\Local\Temp\ipykernel_10504\3986136400.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid['lat'] = grid.geometry.centroid.y


In [6]:
# === 6. 保存最终结果 ===

if 'final_df' in locals() and not final_df.empty:
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n\033[1m结果已成功保存到: {OUTPUT_CSV}\033[0m")
else:
    print("\n警告: 最终DataFrame为空或不存在，未保存任何文件。")


结果已成功保存到: final_environment.csv
